# 2026-012 Natural De-Diff D35 Analysis

This notebook details flow cytometry analysis for 2026-012 De-Diff Day 35

## Initialize Environment

In [14]:
# Load Packages
library(fcexpr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggpubr)
library(ggsci)
library(stringr)
library(jsonlite)

In [15]:
# Set Working Directory
setwd("/home/dalbao/AlbaoRunx3Manuscript/flow/Fig02")

# Load flow processing functions
source("/home/dalbao/AlbaoRunx3Manuscript/flow/scripts/flowProcessing.R")

# Load Workspace (or cached version if it exists)
workspace <- load_workspace_cached(
                wsp_file = "2026-012-NatRechallenge-Analysis.wsp",
                rds_file = "2026-012-NatRechallenge-Analysis.rds",
                overwrite = FALSE # Set to TRUE to reprocess WSP and overwrite cached workspace
        )

# Extract Counts
counts <- workspace[["counts"]]

# Process Gating Data
Cleanup data to report only gates below CD8.

In [16]:
# Filter only samples in "Samples" FlowJoGroup
surface <- counts %>% filter(grepl("Mix", FlowJoGroup))

# Replace MixA in FlowJoGroup as Mix3
surface$FlowJoGroup <- gsub("MixA", "Mix3", surface$FlowJoGroup)

# Replace _CD8 in FileName with ""
surface$SampleID <- paste0(surface$FlowJoGroup, "_", str_extract(surface$FileName, "\\d(?=\\.fcs)"))

head(surface)

,FileName,PopulationFullPath,Parent,Population,Count,ParentCount,FractionOfParent,xDim,yDim,eventsInside,FilePath,FlowJoGroup,ws,SampleID
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
268,Samples_Mix2_001.fcs,root,NA,root,6182667,NA,NA,NA,NA,NA,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
269,Samples_Mix2_001.fcs,Lymphocytes,root,NA/Lymphocytes,5053513,6182667,0.8173678,FSC-A,SSC-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
270,Samples_Mix2_001.fcs,Lymphocytes/Single Cells,Lymphocytes,Lymphocytes/Single Cells,4379196,5053513,0.8665647,FSC-W,FSC-H,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
271,Samples_Mix2_001.fcs,Lymphocytes/Single Cells/Lymphocytes,Lymphocytes/Single Cells,Single Cells/Lymphocytes,4379196,4379196,1.0000000,FSC-A,SSC-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
272,Samples_Mix2_001.fcs,Lymphocytes/Single Cells/Single Cells,Lymphocytes/Single Cells,Single Cells/Single Cells,3985229,4379196,0.9100367,SSC-W,SSC-H,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
273,Samples_Mix2_001.fcs,Lymphocytes/Single Cells/Single Cells/CD8,Lymphocytes/Single Cells/Single Cells,CD8,2115097,3985229,0.5307341,Comp-PE W_561-A,Comp-BUV 396-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1


Process df:

In [17]:
# Keep only observations in surface wherin PopulationFullPath contains
# "primary|SP|DP"
surface <- surface %>%
    filter(grepl("primary|SP|DP", PopulationFullPath))

# Remove "secondary/"
surface <- surface %>%
    mutate(PopulationFullPath = gsub("secondary/", "", PopulationFullPath))

# Remove "Lymphocytes/Single Cells/Single Cells/CD8/" from PopulationFullPath
surface <- surface %>%
    mutate(PopulationFullPath = gsub("Lymphocytes/Single Cells/Single Cells/CD8/", "", PopulationFullPath))

head(surface)

,FileName,PopulationFullPath,Parent,Population,Count,ParentCount,FractionOfParent,xDim,yDim,eventsInside,FilePath,FlowJoGroup,ws,SampleID
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
274,Samples_Mix2_001.fcs,primary,Lymphocytes/Single Cells/Single Cells/CD8,primary,143564,2115097,0.06787585,Comp-PerCP Cy5-5-A,Comp-BUV496-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
277,Samples_Mix2_001.fcs,primary/63-3,Lymphocytes/Single Cells/Single Cells/CD8/primary,primary/63-3,46895,143564,0.32664874,Comp-PE-Cy7-A,Comp-BV510-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
278,Samples_Mix2_001.fcs,primary/63-6,Lymphocytes/Single Cells/Single Cells/CD8/primary,primary/63-6,12476,143564,0.08690201,Comp-PE-Cy7-A,Comp-BV510-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
279,Samples_Mix2_001.fcs,primary/63-63,Lymphocytes/Single Cells/Single Cells/CD8/primary,primary/63-63,40647,143564,0.28312808,Comp-PE-Cy7-A,Comp-BV510-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
280,Samples_Mix2_001.fcs,primary/63-DN,Lymphocytes/Single Cells/Single Cells/CD8/primary,primary/63-DN,43546,143564,0.30332117,Comp-PE-Cy7-A,Comp-BV510-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1
281,Samples_Mix2_001.fcs,primary/A,Lymphocytes/Single Cells/Single Cells/CD8/primary,primary/A,14772,143564,0.10289488,Comp-BV650-A,Comp-PE-Cy7-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-RechallengeD35+7/Samples_Mix2_001.fcs,Mix2,09-Jun-2026.wsp,Mix2_1


In [18]:
# Select relevant columns
surface <- surface %>%
    select(SampleID, FlowJoGroup, PopulationFullPath, Count, FractionOfParent) %>% 
    # Rename FractionOfParent to Percent
    rename(Percent = FractionOfParent, Mix = FlowJoGroup) %>% # mutate
    mutate(Percent = Percent * 100) # Convert to percentage
    # Renme SampleID to ID
surface <- surface %>% rename(ID = SampleID)

# Preview
head(surface)

,ID,Mix,PopulationFullPath,Count,Percent
,<chr>,<chr>,<chr>,<dbl>,<dbl>
274,Mix2_1,Mix2,primary,143564,6.787585
277,Mix2_1,Mix2,primary/63-3,46895,32.664874
278,Mix2_1,Mix2,primary/63-6,12476,8.690201
279,Mix2_1,Mix2,primary/63-63,40647,28.312808
280,Mix2_1,Mix2,primary/63-DN,43546,30.332117
281,Mix2_1,Mix2,primary/A,14772,10.289488


Calculate total percent:

In [19]:
# Without rounding at all
total_percent <- compute_percent_of_total(surface) %>%
    add_population_derivatives() %>%
    finalize_flow_table(round_digits = NULL)

head(total_percent, n = 10)

ID,Population,Population1Deriv,Mix,Count,Percent,PercentOfTotal
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
Mix2_1,DP,NA,Mix2,4140,95.04132231,95.04132231
Mix2_1,DP,63-3,Mix2,201,4.85507246,4.61432507
Mix2_1,DP,63-6,Mix2,872,21.06280193,20.01836547
Mix2_1,DP,63-63,Mix2,222,5.36231884,5.09641873
Mix2_1,DP,63-DN,Mix2,2845,68.71980676,65.31221304
Mix2_1,DP,A,Mix2,1240,29.95169082,28.46648301
Mix2_1,DP,B,Mix2,1629,39.34782609,37.39669421
Mix2_1,DP,C,Mix2,75,1.81159420,1.72176309
Mix2_1,DP,D,Mix2,4,0.09661836,0.09182736


In [20]:
interesting <- total_percent %>%
    filter((grepl("Gerlach", Population1Deriv))) %>%
    mutate(Population1Deriv = factor(gsub("Gerlach-", "", Population1Deriv), levels = c("Lo", "Int", "Hi")))

In [21]:
interesting

ID,Population,Population1Deriv,Mix,Count,Percent,PercentOfTotal
<chr>,<chr>,<fct>,<chr>,<dbl>,<dbl>,<dbl>
Mix2_1,DP,Hi,Mix2,3310,79.9516908,75.9871442
Mix2_1,DP,Int,Mix2,701,16.9323671,16.0927456
Mix2_1,DP,Lo,Mix2,40,0.9661836,0.9182736
Mix2_1,SP,Hi,Mix2,27,28.7234043,0.6198347
Mix2_1,SP,Int,Mix2,41,43.6170213,0.9412305
Mix2_1,SP,Lo,Mix2,10,10.6382979,0.2295684
Mix2_1,primary,Hi,Mix2,29166,20.3156780,1.3789438
Mix2_1,primary,Int,Mix2,77897,54.2594244,3.6829044
Mix2_1,primary,Lo,Mix2,24032,16.7395726,1.1362127


## Plot

In [22]:
# Use theme_bw, base 12, not bold, and remove legend
theme <- theme_bw(
    base_size = 14,
    base_family = "sans"
) +
    theme(
        axis.text = element_text(size = 14, color = "black"),
        axis.title = element_text(size = 14, color = "black"),
        axis.text.x = element_text(angle = 45, hjust = 0.5, vjust = 0.5, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_blank(),

        # Remove facet strip background + box
        strip.background = element_rect(fill = NA, color = NA),
        strip.text = element_text(color = "black"),  # optional: keep text styling clean

        legend.position = "none",
        plot.title = element_text(size = 12),
        panel.grid = element_blank()
    )

In [23]:
comparisons <- list(
    c("SP", "DP"),
    c("SP", "primary"),
    c("DP", "primary")
)

mix_ident <- list(

    Mix2 = c(SP = "b", DP = "c", primary  = "d"),
    Mix3 = c(SP = "c", DP = "a", primary  = "d")

)

cols = c(a = "#1D2B53", b = "#7E2553", c = "#FF004D", d = "#FFFFFF")

In [24]:
for(mix in c("Mix2", "Mix3")){

    # If Population1Deriv is NA, change value to "all"
    plot.data.df <- interesting %>% filter(Mix == mix & Population != "all")

    plot.data.df <- plot.data.df %>% 
        mutate( 
                # Replace SP and DP in Population column with mix_ident values
                Population = factor(    Population, levels = c("SP", "DP", "primary"),
                                        labels = c(mix_ident[[mix]]["SP"], mix_ident[[mix]]["DP"], mix_ident[[mix]]["primary"])
                )
        )

    # Ensure paired samples line up correctly: plot_padj()/rstatix pair rows
    # POSITIONALLY (i-th row of group1 with i-th row of group2 as they appear
    # in the data), NOT by ID. Sorting by ID within each facet guarantees
    # both Population groups are in the same ID order, so position == animal.
    plot.data.df <- plot.data.df %>% arrange("Population1Deriv", ID)

    # comparisons is defined in terms of "SP"/"DP", but Population has just been
    # relabeled to this mix's letter codes above, so translate it through
    # mix_ident too, or plot_padj() filters on labels that no longer exist
    # in the data (empty x/y vectors -> t.test "not enough 'x' observations")
    comparisons_mix <- lapply(comparisons, function(x) unname(mix_ident[[mix]][x]))

    # Calculate p-values and adjusted p-values for comparisons using plot_padj function
    plot.stat.df <- plot_padj(
        data = plot.data.df,
        comparisons = comparisons_mix,
        id_col = "ID",
        y_col = "Percent",
        x_col = "Population",
        facet_col = "Population1Deriv",
        # pool_sd = TRUE,
        paired = TRUE,
        step_fraction = 0.1,
        min_step = 0.5
    )
    # Save plot.stat.df to a CSV file for inspection
    write.csv(plot.stat.df, paste0("2026-012-Fig02g-", mix, "rechallenge.csv"))

    # Round plot.stat.df column foldmean to one significant digit
    # Then concatenate to p.adj.signif only if not "ns", into column p.adj.fold
    plot.stat.df <- plot.stat.df %>%
        mutate(foldmeanround = signif(foldmean, digits = 2)) %>%
        mutate(p.adj.fold = ifelse(p.adj.signif == "ns", "ns", paste0(p.adj.signif, " (", foldmeanround, "x)")))

    # Only the significant comparisons get a bracket drawn (see
    # stat_pvalue_manual below), so only these need extra headroom
    plot.stat.sig <- plot.stat.df %>% filter(p.adj < 0.05)

    # Create the plot
    p <- ggplot(plot.data.df, aes(x = Population, y = Percent, fill = Population)) +
        geom_boxplot(outlier.shape = NA) +
        # Connect each subject's SP and DP points to identify paired samples
        geom_line(aes(group = ID), color = "black", linewidth = 0.5) +
        geom_point(size = 2) +
        facet_wrap(~ Population1Deriv, ncol = 3, scales = "free_y") +
        theme +
        scale_fill_manual(values = cols) +
        # Make room for the p-value bracket: with scales = "free_y", each panel
        # auto-scales to plot.data.df alone, so the bracket drawn at y.position
        # (above the tallest box, per plot_padj()'s step_fraction/min_step) would
        # get clipped at the top of the panel without this. geom_blank() is
        # invisible but still trains the y-scale, and a small multiplier adds
        # headroom for the label text sitting above the bracket line itself.
        geom_blank( data = plot.stat.sig, aes(x = group1, y = y.position),
                    inherit.aes = FALSE) +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.35))) +
        stat_pvalue_manual(
            plot.stat.sig,
            label = "p.adj.fold",   # adjusted p-value stars
            tip.length = 0.05,
            size = 4,
            inherit.aes = FALSE       # important fix
        ) +
        theme (  axis.text.x = element_text(angle = 0),
                    strip.text = element_blank(),
                    strip.background = element_blank())
                
    pdf(file =  paste0("2026-012-Fig02g-", mix, "rechallenge.pdf"), width = 4, height = 1.7)
    print(p)
    dev.off()
}

In [25]:
combineMixes <- interesting %>% filter(Population != "all") %>%
    mutate(Group =  ifelse(Population == "primary", "d",
                        ifelse(Mix == "Mix2" & Population == "SP", "b",
                            ifelse(Mix == "Mix3" & Population == "DP", "a", "c")
                        )
                    )) %>% mutate(Group = factor(Group, levels = c("a", "b", "c", "d")))
combineMixes

ID,Population,Population1Deriv,Mix,Count,Percent,PercentOfTotal,Group
<chr>,<chr>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<fct>
Mix2_1,DP,Hi,Mix2,3310,79.9516908,75.9871442,c
Mix2_1,DP,Int,Mix2,701,16.9323671,16.0927456,c
Mix2_1,DP,Lo,Mix2,40,0.9661836,0.9182736,c
Mix2_1,SP,Hi,Mix2,27,28.7234043,0.6198347,b
Mix2_1,SP,Int,Mix2,41,43.6170213,0.9412305,b
Mix2_1,SP,Lo,Mix2,10,10.6382979,0.2295684,b
Mix2_1,primary,Hi,Mix2,29166,20.3156780,1.3789438,d
Mix2_1,primary,Int,Mix2,77897,54.2594244,3.6829044,d
Mix2_1,primary,Lo,Mix2,24032,16.7395726,1.1362127,d


In [28]:
comparisons <- list(
    c("a", "b"),
    c("a", "c"),
    c("a", "d"),
    c("b", "c"),
    c("b", "d"),
    c("c", "d")
)

# Calculate p-values and adjusted p-values for comparisons using plot_padj function
plot.stat.df <- plot_padj(
    data = combineMixes,
    comparisons = comparisons,
    y_col = "Percent",
    x_col = "Group",
    facet_col = "Population1Deriv",
    paired = FALSE,
    step_fraction = 0.1,
    min_step = 0.5
)

# Save plot.stat.df to a CSV file for inspection
write.csv(plot.stat.df, paste0("2026-012-EFig04-rechallenge.csv"))

# Round plot.stat.df column foldmean to one significant digit
# Then concatenate to p.adj.signif only if not "ns", into column p.adj.fold
plot.stat.df <- plot.stat.df %>%
    mutate(foldmeanround = signif(foldmean, digits = 2)) %>%
    mutate(p.adj.fold = ifelse(p.adj.signif == "ns", "ns", paste0(p.adj.signif, " (", foldmeanround, "x)")))

# Only the significant comparisons get a bracket drawn (see
# stat_pvalue_manual below), so only these need extra headroom
plot.stat.sig <- plot.stat.df %>% filter(p.adj < 0.05)


# Create the plot
p <- ggplot(combineMixes, aes(x = Group, y = Percent, fill = Group)) +
    geom_boxplot(outlier.shape = NA) +
    # Connect each subject's SP and DP points to identify paired samples
    geom_line(aes(group = ID), color = "black", linewidth = 0.5) +
    geom_point(size = 2) +
    facet_wrap(~ Population1Deriv, ncol = 3, scales = "free_y") +
    theme +
    scale_fill_manual(values = cols) +
    # Make room for the p-value bracket: with scales = "free_y", each panel
    # auto-scales to plot.data.df alone, so the bracket drawn at y.position
    # (above the tallest box, per plot_padj()'s step_fraction/min_step) would
    # get clipped at the top of the panel without this. geom_blank() is
    # invisible but still trains the y-scale, and a small multiplier adds
    # headroom for the label text sitting above the bracket line itself.
    geom_blank( data = plot.stat.sig, aes(x = group1, y = y.position),
                inherit.aes = FALSE) +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.35))) +
    stat_pvalue_manual(
        plot.stat.sig,
        label = "p.adj.fold",   # adjusted p-value stars
        tip.length = 0.05,
        size = 4,
        inherit.aes = FALSE       # important fix
    ) +
    theme (  axis.text.x = element_text(angle = 0),
                strip.text = element_blank(),
                strip.background = element_blank())
            
pdf(file =  paste0("2026-012-EFig04-rechallenge.pdf"), width = 4, height = 1.7)
print(p)
dev.off()

pdf 
  2

In [12]:
interesting <- total_percent %>%
    filter(is.na(Population1Deriv) & Population != "primary")

interesting

ID,Population,Population1Deriv,Mix,Count,Percent,PercentOfTotal
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
Mix2_1,DP,NA,Mix2,4140,95.0413223,95.0413223
Mix2_1,SP,NA,Mix2,94,2.1579431,2.1579431
Mix2_2,DP,NA,Mix2,1935,54.7848245,54.7848245
Mix2_2,SP,NA,Mix2,1558,44.1109853,44.1109853
Mix2_3,DP,NA,Mix2,1296,57.6769025,57.6769025
Mix2_3,SP,NA,Mix2,925,41.1659991,41.1659991
Mix2_4,DP,NA,Mix2,12214,46.0836100,46.0836100
Mix2_4,SP,NA,Mix2,14005,52.8410806,52.8410806
Mix3_1,DP,NA,Mix3,3170,95.5394816,95.5394816


In [13]:
comparisons <- list(
    c("SP", "DP")
)

mix_ident <- list(

    Mix2 = c(SP = "b", DP = "c"),
    Mix3 = c(SP = "c", DP = "a")

)

cols = c(a = "#1D2B53", b = "#7E2553", c = "#FF004D", d = "#FFFFFF")

In [14]:
for(mix in c("Mix2", "Mix3")){

    # If Population1Deriv is NA, change value to "all"
    plot.data.df <- interesting %>% filter(Mix == mix)

    plot.data.df <- plot.data.df %>% 
        mutate( 
                # Replace SP and DP in Population column with mix_ident values
                Population = factor(    Population, levels = c("SP", "DP"),
                                        labels = c(mix_ident[[mix]]["SP"], mix_ident[[mix]]["DP"])
                )
        )

    # Ensure paired samples line up correctly: plot_padj()/rstatix pair rows
    # POSITIONALLY (i-th row of group1 with i-th row of group2 as they appear
    # in the data), NOT by ID. Sorting by ID within each facet guarantees
    # both Population groups are in the same ID order, so position == animal.
    plot.data.df <- plot.data.df %>% arrange("Population1Deriv", ID)

    # comparisons is defined in terms of "SP"/"DP", but Population has just been
    # relabeled to this mix's letter codes above, so translate it through
    # mix_ident too, or plot_padj() filters on labels that no longer exist
    # in the data (empty x/y vectors -> t.test "not enough 'x' observations")
    comparisons_mix <- lapply(comparisons, function(x) unname(mix_ident[[mix]][x]))

    # Calculate p-values and adjusted p-values for comparisons using plot_padj function
    plot.stat.df <- plot_padj(
        data = plot.data.df,
        comparisons = comparisons_mix,
        id_col = "ID",
        y_col = "Percent",
        x_col = "Population",
        facet_col = "Population1Deriv",
        # pool_sd = TRUE,
        paired = FALSE,
        step_fraction = 0.1,
        min_step = 0.5
    )
    # Save plot.stat.df to a CSV file for inspection
    write.csv(plot.stat.df, paste0("2026-012-Fig02g-", mix, ".ratio_rechallenge.csv"))

    # Round plot.stat.df column foldmean to one significant digit
    # Then concatenate to p.adj.signif only if not "ns", into column p.adj.fold
    plot.stat.df <- plot.stat.df %>%
        mutate(foldmeanround = signif(foldmean, digits = 2)) %>%
        mutate(p.adj.fold = ifelse(p.adj.signif == "ns", "ns", paste0(p.adj.signif, " (", foldmeanround, "x)")))

    # Only the significant comparisons get a bracket drawn (see
    # stat_pvalue_manual below), so only these need extra headroom
    plot.stat.sig <- plot.stat.df %>% filter(p.adj < 0.05)

    # Create the plot
    p <- ggplot(plot.data.df, aes(x = Population, y = Percent, fill = Population)) +
        geom_boxplot(outlier.shape = NA) +
        # Connect each subject's SP and DP points to identify paired samples
        geom_line(aes(group = ID), color = "black", linewidth = 0.5) +
        geom_point(size = 2) +
        facet_wrap(~ Population1Deriv, ncol = 3, scales = "free_y") +
        theme +
        scale_fill_manual(values = cols) +
        # Make room for the p-value bracket: with scales = "free_y", each panel
        # auto-scales to plot.data.df alone, so the bracket drawn at y.position
        # (above the tallest box, per plot_padj()'s step_fraction/min_step) would
        # get clipped at the top of the panel without this. geom_blank() is
        # invisible but still trains the y-scale, and a small multiplier adds
        # headroom for the label text sitting above the bracket line itself.
        geom_blank( data = plot.stat.sig, aes(x = group1, y = y.position),
                    inherit.aes = FALSE) +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.35))) +
        stat_pvalue_manual(
            plot.stat.sig,
            label = "p.adj.fold",   # adjusted p-value stars
            tip.length = 0.05,
            size = 4,
            inherit.aes = FALSE       # important fix
        ) +
        theme (  axis.text.x = element_text(angle = 0),
                    strip.text = element_blank(),
                    strip.background = element_blank())
                
    pdf(file = paste0("2026-012-Fig02g-", mix, ".ratio_rechallenge.pdf"), width = 1.5, height = 1.7)
    print(p)
    dev.off()
}